In [2]:
%pip install kagglehub --quiet
import kagglehub
import tqdm as notebook_tqdm
from pathlib import Path

ds1_path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
ds2_path = kagglehub.dataset_download("niyarrbarman/symptom2disease")

print("Dataset 1 root:", ds1_path)
print("Dataset 2 root:", ds2_path)


def _find_csv(root: str, hint: str) -> str:
    """Pick the CSV under `root` whose name contains `hint` (case-insensitive)."""
    candidates = [p for p in Path(root).rglob("*.csv")]
    if not candidates:
        raise FileNotFoundError(f"No CSV under {root}")
    matches = [p for p in candidates if hint.lower() in p.name.lower()]
    chosen = matches[0] if matches else candidates[0]
    return str(chosen)


DS1_CSV = _find_csv(ds1_path, "Final_Augmented")
DS2_CSV = _find_csv(ds2_path, "Symptom2Disease")
print("DS1 CSV:", DS1_CSV)
print("DS2 CSV:", DS2_CSV)


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset 1 root: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\dhivyeshrk\diseases-and-symptoms-dataset\versions\1
Dataset 2 root: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\niyarrbarman\symptom2disease\versions\1
DS1 CSV: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\dhivyeshrk\diseases-and-symptoms-dataset\versions\1\Final_Augmented_dataset_Diseases_and_Symptoms.csv
DS2 CSV: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\niyarrbarman\symptom2disease\versions\1\Symptom2Disease.csv


In [ ]:
%pip install pandas --quiet
import pandas as pd

df_ds1 = pd.read_csv(DS1_CSV)
display(df_ds1.head())
display(df_ds1["diseases"].value_counts())


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Columns: ['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness', 'palpitations', 'irregular heartbeat', 'breathing fast', 'hoarse voice', 'sore throat', 'difficulty speaking', 'cough', 'nasal congestion', 'throat swelling', 'diminished hearing', 'lump in throat', 'throat feels tight', 'difficulty in swallowing', 'skin swelling', 'retention of urine', 'groin mass', 'leg pain', 'hip pain', 'suprapubic pain', 'blood in stool', 'lack of growth', 'emotional symptoms', 'elbow weakness', 'back weakness', 'pus in sputum', 'symptoms of the scrotum and testes', 'swelling of scrotum', 'pain in testicles', 'flatulence', 'pus draining from ear', 'jaundice', 'mass in scrotum', 'white discharge from eye', 'irritable infant', 'abusing alcohol', 'fainting', 'hostile behavior', 'drug ab

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


diseases
cystitis                          1219
vulvodynia                        1218
nose disorder                     1218
complex regional pain syndrome    1217
spondylosis                       1216
                                  ... 
foreign body in the nose             1
thalassemia                          1
open wound of the head               1
rocky mountain spotted fever         1
kaposi sarcoma                       1
Name: count, Length: 773, dtype: int64

In [4]:
%pip install numpy --quiet
import re
import numpy as np

def _prettify_symptom(col: str) -> str:
    """Convert e.g. 'shortness_of_breath' -> 'shortness of breath'."""
    return re.sub(r"[_\-]+", " ", col).strip().lower()


def build_ds1(df: pd.DataFrame) -> pd.DataFrame:
    # First column is the disease label; the rest are 0/1 symptom flags.
    disease_col = df.columns[0]
    symptom_cols = [c for c in df.columns[1:]]
    pretty = {c: _prettify_symptom(c) for c in symptom_cols}

    # Vectorised: for each row collect the symptom column names where value == 1.
    sym_matrix = df[symptom_cols].to_numpy(dtype=np.int8)
    rows = []
    for i, disease in enumerate(df[disease_col].astype(str).values):
        active = np.flatnonzero(sym_matrix[i] == 1)
        if active.size == 0:
            continue
        symptoms = ", ".join(pretty[symptom_cols[j]] for j in active)
        rows.append({
            "input":  symptoms,
            "output": disease.strip(),
            "source": "ds1_diseases_symptoms",
        })

    out = pd.DataFrame(rows)
    
    # Drop diseases with <2 rows so stratified split is feasible later.
    counts = out["output"].value_counts()
    keep = counts[counts >= 2].index
    out = out[out["output"].isin(keep)].reset_index(drop=True)
    
    return out


ds1 = build_ds1(df_ds1)
print(f"DS1 rows: {len(ds1):,}  |  unique diseases: {ds1['output'].nunique()}")
ds1.head(3)
ds1["output"].value_counts()



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
DS1 rows: 246,926  |  unique diseases: 754


output
cystitis                                        1219
vulvodynia                                      1218
nose disorder                                   1218
complex regional pain syndrome                  1217
spondylosis                                     1216
                                                ... 
diabetic kidney disease                            2
human immunodeficiency virus infection (hiv)       2
carcinoid syndrome                                 2
rheumatic fever                                    2
open wound of the jaw                              2
Name: count, Length: 754, dtype: int64

In [ ]:
df_ds2 = pd.read_csv(DS2_CSV)
#drop unnamed cols
df_ds2 = df_ds2.loc[:, ~df_ds2.columns.str.contains('^Unnamed')]
display(df_ds2.head())
display(df_ds2["label"].value_counts())

Columns: ['label', 'text']
Types: label    str
text     str
dtype: object


,label,text
0,Psoriasis,I have been experiencing a skin rash on my arm...
1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,Psoriasis,I have been experiencing joint pain in my fing...
3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,Psoriasis,"My nails have small dents or pits in them, and..."


label
Psoriasis                          50
Varicose Veins                     50
Typhoid                            50
Chicken pox                        50
Impetigo                           50
Dengue                             50
Fungal infection                   50
Common Cold                        50
Pneumonia                          50
Dimorphic Hemorrhoids              50
Arthritis                          50
Acne                               50
Bronchial Asthma                   50
Hypertension                       50
Migraine                           50
Cervical spondylosis               50
Jaundice                           50
Malaria                            50
urinary tract infection            50
allergy                            50
gastroesophageal reflux disease    50
drug reaction                      50
peptic ulcer disease               50
diabetes                           50
Name: count, dtype: int64

In [ ]:
def build_ds2(df: pd.DataFrame) -> pd.DataFrame:    
    df = df.rename(columns={c: c.lower() for c in df.columns})
    assert {"label", "text"}.issubset(df.columns), f"Unexpected columns: {df.columns.tolist()}"
    df = df.dropna(subset=["label", "text"])
    out = pd.DataFrame({
        "input":  df["text"].astype(str).str.strip().values,
        "output": df["label"].astype(str).str.strip().values,
        "source": "ds2_symptom2disease",
    })
    return out


ds2 = build_ds2(df_ds2)
print(f"DS2 rows: {len(ds2):,}  |  unique diseases: {ds2['output'].nunique()}")
ds2.head(3)

DS2 rows: 1,200  |  unique diseases: 24


output
Psoriasis                          50
Varicose Veins                     50
Typhoid                            50
Chicken pox                        50
Impetigo                           50
Dengue                             50
Fungal infection                   50
Common Cold                        50
Pneumonia                          50
Dimorphic Hemorrhoids              50
Arthritis                          50
Acne                               50
Bronchial Asthma                   50
Hypertension                       50
Migraine                           50
Cervical spondylosis               50
Jaundice                           50
Malaria                            50
urinary tract infection            50
allergy                            50
gastroesophageal reflux disease    50
drug reaction                      50
peptic ulcer disease               50
diabetes                           50
Name: count, dtype: int64

In [ ]:
concat = pd.concat([ds1, ds2], ignore_index=True).reset_index(drop=True)
# normalize output by lowercasing
concat["output"] = concat["output"].str.lower()
print(f"Combined rows: {len(concat):,}  |  unique diseases: {concat['output'].nunique()}")
print("Stable processing index prepared for resumable batching.")

Combined rows: 248,126  |  unique diseases: 767


In [10]:
concat["output"].value_counts()

output
pneumonia                                       1262
cystitis                                        1219
vulvodynia                                      1218
nose disorder                                   1218
complex regional pain syndrome                  1217
                                                ... 
diabetic kidney disease                            2
human immunodeficiency virus infection (hiv)       2
carcinoid syndrome                                 2
rheumatic fever                                    2
open wound of the jaw                              2
Name: count, Length: 767, dtype: int64

In [ ]:
# call llm using Foundry Local's OpenAI-compatible API to validate if the symptoms match the disease label
%pip install openai requests --quiet

import datetime as dt
import json
from typing import Any
import requests
from openai import OpenAI

def _normalize_foundry_base_url(url: str) -> str:
    url = url.strip().rstrip("/")
    return url if url.endswith("/v1") else f"{url}/v1"


FOUNDRY_BASE_URL = _normalize_foundry_base_url(
    os.getenv("FOUNDRY_LOCAL_BASE_URL", "http://localhost:5272/v1")
)
FOUNDRY_HTTP_ROOT = FOUNDRY_BASE_URL.rsplit("/v1", 1)[0]
FOUNDRY_TIMEOUT_SECONDS = 120
FOUNDRY_TEMPERATURE = 0.1
FOUNDRY_MAX_TOKENS = 450

SMOKE_TEST_SIZE = 3
SMOKE_TEST_SEED = 42
BATCH_SIZE = 3
RESUME_MODE = True
PROGRESS_LOG_PATH = Path("augmentation_progress.log")
PARTIAL_DATASET_PATH = Path("combined_diseases_symptoms.partial.csv")
FINAL_DATASET_PATH = Path("combined_diseases_symptoms.csv")

foundry_client = OpenAI(
    base_url=FOUNDRY_BASE_URL,
    api_key="not-needed-for-local",
    timeout=FOUNDRY_TIMEOUT_SECONDS,
    max_retries=2,
)


def _extract_json_object(text: str) -> dict[str, Any]:
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty response from Foundry Local")

    fenced_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if fenced_match:
        text = fenced_match.group(1).strip()
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            text = text[start:end + 1]

    return json.loads(text)


def _get_foundry_loaded_models() -> list[str]:
    response = requests.get(
        f"{FOUNDRY_HTTP_ROOT}/openai/loadedmodels",
        timeout=15,
    )
    response.raise_for_status()
    models = response.json()
    if not isinstance(models, list):
        raise ValueError(f"Unexpected model list payload: {models}")
    return [str(model).strip() for model in models if str(model).strip()]


def _get_foundry_available_models() -> list[str]:
    response = requests.get(
        f"{FOUNDRY_HTTP_ROOT}/openai/models",
        timeout=15,
    )
    response.raise_for_status()
    models = response.json()
    if not isinstance(models, list):
        raise ValueError(f"Unexpected model list payload: {models}")
    return [str(model).strip() for model in models if str(model).strip()]


def _resolve_foundry_model() -> str:
    preferred_model = os.getenv("FOUNDRY_MODEL", "qwen2.5-7b-instruct-openvino-npu:3").strip()
    if preferred_model:
        return preferred_model

    loaded_models = _get_foundry_loaded_models()
    if loaded_models:
        return loaded_models[0]

    available_models = _get_foundry_available_models()
    if available_models:
        return available_models[0]

    raise RuntimeError(
        "No model was found in Foundry Local. Load a model first or set FOUNDRY_MODEL explicitly."
    )


FOUNDRY_MODEL = _resolve_foundry_model()
print(f"Foundry Local endpoint: {FOUNDRY_BASE_URL}")
print(f"Foundry model: {FOUNDRY_MODEL}")
print(f"Progress log: {PROGRESS_LOG_PATH.resolve()}")


def _normalize_candidate_diseases(disease: str, supported: bool, alternatives: list[str] | None) -> list[str]:
    cleaned = []
    seen = set()
    for value in alternatives or []:
        candidate = str(value).strip().lower()
        if candidate and candidate not in seen:
            cleaned.append(candidate)
            seen.add(candidate)

    disease = (disease or "").strip().lower()
    if supported and disease:
        return [disease]

    return cleaned


def _format_reasoning_output(reasoning: str, supported: bool, disease: str, alternatives: list[str] | None = None) -> str:
    reasoning = (reasoning or "The available symptom pattern is not sufficient for a fully reliable clinical conclusion.").strip()
    disease = (disease or "").strip().lower()
    alternatives = _normalize_candidate_diseases(disease, supported, alternatives)

    if supported and disease:
        conclusion = f"Based on the reported symptoms, the clinical indication points to: {disease}."
    else:
        if alternatives:
            if len(alternatives) == 1:
                alternative_text = alternatives[0]
            elif len(alternatives) == 2:
                alternative_text = " or ".join(alternatives)
            else:
                alternative_text = ", ".join(alternatives[:-1]) + f", or {alternatives[-1]}"
            conclusion = (
                "Based on the reported symptoms, it's not possible to affirm which disease it is for sure, "
                f"it may be {alternative_text}."
            )
        else:
            conclusion = (
                "Based on the reported symptoms, it's not possible to affirm which disease it is for sure, "
                "and the available symptom set remains clinically inconclusive."
            )

    disclaimer = (
        "Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. "
        "It is not 100% precise and does not replace a professional medical diagnosis."
    )

    return f"<think>{reasoning}</think>\n\n{conclusion}\n{disclaimer}"


def validate_with_llm(row, *, emit_console: bool = True) -> dict[str, Any]:
    symptoms = str(row["input"]).strip()
    disease = str(row["output"]).strip().lower()
    source = str(row["source"]).strip()

    system_prompt = """You are a clinical dataset validation assistant.
Assess whether the reported symptoms strongly support the provided disease label.
Return valid JSON only, without markdown fences or extra commentary.
Use this schema exactly:
{
  \"reasoning\": \"brief clinical reasoning\",
  \"supported\": true,
  \"alternatives\": [\"alternative 1\", \"alternative 2\"]
}
Rules:
- Set supported=true only when the symptoms support the supplied disease label with reasonable confidence.
- Set supported=false when the symptom pattern is insufficient, ambiguous, or more compatible with other diseases.
- When supported=true, alternatives should usually be an empty list.
- When supported=false, include up to 3 concise differential diagnoses when appropriate.
- Do not mention that you are an AI model.
"""

    user_prompt = f"""Symptoms reported: {symptoms}
Disease label under review: {disease}

Decide whether the symptoms allow identifying this disease with confidence, or whether the case is inconclusive.
"""

    error_message = None

    try:
        response = foundry_client.chat.completions.create(
            model=FOUNDRY_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=FOUNDRY_TEMPERATURE,
            max_tokens=FOUNDRY_MAX_TOKENS,
        )
        raw_content = response.choices[0].message.content or ""
        parsed = _extract_json_object(raw_content)

        reasoning = str(parsed.get("reasoning", "")).strip()
        supported = bool(parsed.get("supported", False))
        alternatives = parsed.get("alternatives") or []
        if not isinstance(alternatives, list):
            alternatives = [str(alternatives)]

    except Exception as exc:
        error_message = str(exc)
        print(f"Foundry Local validation failed for '{disease}': {exc}")
        reasoning = (
            "The symptom description could not be validated with enough confidence during this pass, "
            "so the case should be treated as inconclusive."
        )
        supported = False
        alternatives = []

    candidate_diseases = _normalize_candidate_diseases(disease, supported, alternatives)
    support_status = "supported" if supported else "inconclusive"
    final_text = _format_reasoning_output(
        reasoning=reasoning,
        supported=supported,
        disease=disease,
        alternatives=candidate_diseases,
    )

    result = {
        "input": symptoms,
        "output": disease,
        "source": source,
        "support_status": support_status,
        "candidate_diseases": candidate_diseases,
        "reasoning": final_text,
        "raw_reasoning": reasoning,
        "model": FOUNDRY_MODEL,
        "error": error_message,
        "processed_at": dt.datetime.now(dt.UTC).isoformat(),
    }

    if emit_console:
        print("=" * 100)
        print(f"Symptoms: {symptoms}")
        print(f"Label: {disease}")
        print(f"Support status: {support_status}")
        print(f"Candidate diseases: {candidate_diseases}")
        print(final_text)

    return result


def _payload_to_progress_line(index: int, payload: dict[str, Any]) -> str:
    return f"{index}: {json.dumps(payload, ensure_ascii=False)}\n"


def _load_progress_entries(path: Path) -> dict[int, dict[str, Any]]:
    entries: dict[int, dict[str, Any]] = {}
    if not path.exists():
        return entries

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                index_text, payload_text = stripped.split(":", 1)
                index = int(index_text.strip())
                payload = json.loads(payload_text.strip())
                if not isinstance(payload, dict):
                    raise ValueError("Progress payload must be a JSON object.")
                if not isinstance(payload.get("candidate_diseases", []), list):
                    payload["candidate_diseases"] = []
                entries[index] = payload
            except Exception as exc:
                print(f"Warning: skipped malformed progress line {line_number}: {exc}")

    return entries


def _append_progress_entry(path: Path, index: int, payload: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(_payload_to_progress_line(index, payload))
        handle.flush()


def _reconstruct_enriched_concat(
    base_df: pd.DataFrame,
    progress_entries: dict[int, dict[str, Any]],
    *,
    require_complete: bool,
    required_columns: list[str] | None = None,
    missing_fill: Any = None,
    missing_candidate_value: str = "[]",
) -> pd.DataFrame:
    required_columns = required_columns or ["reasoning", "candidate_diseases", "support_status"]
    missing_indices = sorted(set(base_df.index) - set(progress_entries))
    if require_complete and missing_indices:
        preview = missing_indices[:10]
        raise ValueError(
            f"Missing progress entries for {len(missing_indices)} row(s). First missing indices: {preview}"
        )

    reconstructed = base_df[["input", "output", "source"]].copy()
    reconstructed["reasoning"] = [
        progress_entries.get(index, {}).get("reasoning", missing_fill)
        for index in reconstructed.index
    ]
    reconstructed["candidate_diseases"] = [
        json.dumps(progress_entries.get(index, {}).get("candidate_diseases", []), ensure_ascii=False)
        if index in progress_entries else missing_candidate_value
        for index in reconstructed.index
    ]
    reconstructed["support_status"] = [
        progress_entries.get(index, {}).get("support_status", missing_fill)
        for index in reconstructed.index
    ]

    return reconstructed


def _save_partial_checkpoint(base_df: pd.DataFrame, progress_entries: dict[int, dict[str, Any]], path: Path) -> None:
    partial_df = _reconstruct_enriched_concat(
        base_df,
        progress_entries,
        require_complete=False,
        missing_fill="",
        missing_candidate_value="[]",
    )
    partial_df.to_csv(path, index=False)


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Foundry Local endpoint: http://localhost:5272/v1
Foundry model: qwen2.5-7b-instruct-openvino-npu:3


In [ ]:
# Smoke test with 3 random examples (does not write to the resumable progress log).
smoke_sample = concat.sample(
    n=min(SMOKE_TEST_SIZE, len(concat)),
    random_state=SMOKE_TEST_SEED,
).sort_index()

smoke_results = []
for sample_index, row in smoke_sample.iterrows():
    result = validate_with_llm(row, emit_console=True)
    smoke_results.append({
        "index": int(sample_index),
        "output": result["output"],
        "source": result["source"],
        "support_status": result["support_status"],
        "candidate_diseases": json.dumps(result["candidate_diseases"], ensure_ascii=False),
    })

smoke_results_df = pd.DataFrame(smoke_results)
print(f"Smoke test completed with {len(smoke_results_df)} example(s).")
display(smoke_results_df)

In [ ]:
# Process the dataset in resumable batches and persist progress after every row and batch.
if not RESUME_MODE and PROGRESS_LOG_PATH.exists():
    PROGRESS_LOG_PATH.unlink()
    print(f"Removed existing progress log at {PROGRESS_LOG_PATH} because RESUME_MODE is disabled.")

progress_entries = _load_progress_entries(PROGRESS_LOG_PATH) if RESUME_MODE else {}
completed_indices = set(progress_entries)
print(f"Loaded {len(completed_indices)} completed row(s) from the progress log.")

for batch_start in range(0, len(concat), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(concat))
    batch_indices = list(range(batch_start, batch_end))
    pending_indices = [index for index in batch_indices if index not in completed_indices]

    if not pending_indices:
        print(f"Batch {batch_start}:{batch_end} already completed. Skipping Foundry calls.")
        _save_partial_checkpoint(concat, progress_entries, PARTIAL_DATASET_PATH)
        continue

    print(f"Processing batch {batch_start}:{batch_end} with pending rows {pending_indices}.")
    for index in notebook_tqdm.tqdm(
        pending_indices,
        total=len(pending_indices),
        desc=f"Batch {batch_start}:{batch_end}",
    ):
        row = concat.iloc[index]
        result = validate_with_llm(row, emit_console=True)
        payload = {
            "input": result["input"],
            "output": result["output"],
            "source": result["source"],
            "support_status": result["support_status"],
            "candidate_diseases": result["candidate_diseases"],
            "reasoning": result["reasoning"],
            "raw_reasoning": result["raw_reasoning"],
            "model": result["model"],
            "error": result["error"],
            "processed_at": result["processed_at"],
        }
        _append_progress_entry(PROGRESS_LOG_PATH, index, payload)
        progress_entries[index] = payload
        completed_indices.add(index)

    _save_partial_checkpoint(concat, progress_entries, PARTIAL_DATASET_PATH)
    print(
        f"Saved partial checkpoint after batch {batch_start}:{batch_end}. "
        f"Completed rows so far: {len(completed_indices)}/{len(concat)}"
    )

Generating Foundry Local reasoning:   0%|          | 1/248126 [00:34<2395:07:41, 34.75s/it]

Symptoms: anxiety and nervousness, shortness of breath, depressive or psychotic symptoms, chest tightness, palpitations, irregular heartbeat, breathing fast
Label: panic disorder
<think>The reported symptoms align well with a panic attack, which is a key component of panic disorder. However, some symptoms like depressive or psychotic symptoms, and irregular heartbeat could also suggest other conditions.</think>

Based on the reported symptoms, the clinical indication points to: panic disorder.
Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. It is not 100% precise and does not replace a professional medical diagnosis.


Generating Foundry Local reasoning:   0%|          | 2/248126 [00:56<1874:02:20, 27.19s/it]

Symptoms: shortness of breath, depressive or psychotic symptoms, dizziness, insomnia, palpitations
Label: panic disorder
<think>The symptoms include shortness of breath, dizziness, and palpitations which can be associated with panic disorder. However, depressive or psychotic symptoms and insomnia are more indicative of mood disorders like depression or bipolar disorder.</think>

Based on the reported symptoms, it's not possible to affirm which disease it is for sure, it may be depression or bipolar disorder.
Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. It is not 100% precise and does not replace a professional medical diagnosis.


Generating Foundry Local reasoning:   0%|          | 3/248126 [01:17<1667:14:49, 24.19s/it]

...

Symptoms: anxiety and nervousness, depression, shortness of breath, depressive or psychotic symptoms, abnormal involuntary movements, chest tightness, breathing fast
Label: panic disorder
<think>The reported symptoms include anxiety, nervousness, shortness of breath, chest tightness, and breathing fast, which are common in panic disorder. However, the presence of depressive or psychotic symptoms and abnormal involuntary movements suggests a broader differential diagnosis.</think>

Based on the reported symptoms, it's not possible to affirm which disease it is for sure, it may be depression or schizophrenia.
Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. It is not 100% precise and does not replace a professional medical diagnosis.


Generating Foundry Local reasoning:   0%|          | 403/248126 [7:44:16<1464:58:58, 21.29s/it]

Symptoms: depressive or psychotic symptoms, dizziness, abnormal involuntary movements, irregular heartbeat, breathing fast
Label: panic disorder
<think>The reported symptoms (depressive or psychotic symptoms, dizziness, abnormal involuntary movements, irregular heartbeat, and breathing fast) do not strongly suggest panic disorder. These symptoms could be indicative of various conditions including anxiety disorders, neurological conditions, or cardiovascular issues.</think>

Based on the reported symptoms, it's not possible to affirm which disease it is for sure, it may be anxiety disorder, neurological condition, or cardiovascular issue.
Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. It is not 100% precise and does not replace a professional medical diagnosis.


Generating Foundry Local reasoning:   0%|          | 404/248126 [7:44:39<1497:51:32, 21.77s/it]

Symptoms: depression, dizziness, abnormal involuntary movements, chest tightness, palpitations, irregular heartbeat
Label: panic disorder
<think>The symptoms include dizziness, chest tightness, palpitations, and irregular heartbeat, which can be associated with panic attacks. However, abnormal involuntary movements are less common in panic disorder and more suggestive of conditions like Tourette's syndrome or dystonia.</think>

Based on the reported symptoms, it's not possible to affirm which disease it is for sure, it may be tourette's syndrome, dystonia, or generalized anxiety disorder.
Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. It is not 100% precise and does not replace a professional medical diagnosis.


In [ ]:
# Reconstruct the final dataset from the progress log, validate it, and save the enriched concat.
progress_entries = _load_progress_entries(PROGRESS_LOG_PATH)
concat_final = _reconstruct_enriched_concat(
    concat,
    progress_entries,
    require_complete=True,
)

required_disclaimer = (
    "Disclaimer: This is an AI auxiliary tool designed for healthcare professionals. "
    "It is not 100% precise and does not replace a professional medical diagnosis."
)

assert concat_final[["input", "output", "source"]].equals(
    concat[["input", "output", "source"]]
), "The reconstructed dataset changed the original input/output/source columns."
assert len(concat_final) == len(concat), "The reconstructed dataset row count does not match concat."
assert concat_final["reasoning"].fillna("").str.startswith("<think>").all(), "All rows must start with a <think> block."
assert concat_final["reasoning"].fillna("").str.contains(required_disclaimer, regex=False).all(), "All rows must contain the required disclaimer."
assert concat_final["support_status"].isin(["supported", "inconclusive"]).all(), "Unexpected support status found in the reconstructed dataset."

concat = concat_final.copy()
concat.to_csv(FINAL_DATASET_PATH, index=False)

print(f"Saved {len(concat):,} enriched rows to {FINAL_DATASET_PATH}")
display(concat[["input", "output", "source", "support_status", "candidate_diseases", "reasoning"]].head(3))

## Future integration notes for `screening_robot.ipynb`

This notebook now produces an enriched `concat` dataset that preserves `input`, `output`, and `source`, and appends the following columns:

- `reasoning`: final Qwen-style training text with `<think>...</think>`, conclusion sentence, and disclaimer
- `candidate_diseases`: JSON-encoded list of candidate diseases for the row
- `support_status`: either `supported` or `inconclusive`

### Intended downstream contract

When `screening_robot.ipynb` is updated in the future, the preferred integration path is:

1. keep `extract_disease(text: str)` focused on parsing the model output text itself
2. add a row-aware validator that compares the extracted disease against `candidate_diseases` and `support_status`
3. allow training helpers to optionally consume preformatted `reasoning` from the enriched dataset instead of always regenerating the assistant response from `output` alone
4. keep headline classification metrics comparable by reporting them on rows where `support_status == "supported"`
5. report inconclusive behavior separately, for example with an inconclusive-rate metric and candidate-list analysis

### Suggested future updates

- review how `format_assistant(...)` and `to_chat_record(...)` should behave when `reasoning` is already present
- review how `extract_disease(text: str)` should interact with `candidate_diseases` during validation
- review how `compute_metrics(...)`, confusion matrices, and qualitative comparisons should treat `inconclusive` rows
- prefer machine-readable parsing of `candidate_diseases` with `json.loads(...)` instead of regex extraction from prose

### Notes on storage

- `candidate_diseases` is stored as a JSON-encoded list in the final CSV so it can be decoded later
- `support_status` is the authoritative marker for whether the original label was supported by the Foundry validation step
- the final dataset is reconstructed from `augmentation_progress.log`, so resumability and auditability are part of the intended workflow